# Stadium egress: quick start

Run one S1 feedback pair, evaluate it and inspect a figure. These cells call the same programs as the README. Install `requirements.txt` and make Node.js available before running. Use a new output directory each time.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import os
import shutil
import subprocess
import sys

root = Path.cwd().resolve()
assert (root / "run.py").is_file(), "Open this notebook from the stadium-egress directory."
node = os.environ.get("NODE_BINARY") or shutil.which("node")
assert node, "Set NODE_BINARY to your Node executable or put Node on PATH."
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
run_dir = root / "outputs" / ("notebook_s1_" + stamp)
eval_dir = run_dir.with_name(run_dir.name + "_eval")
figure_dir = run_dir.with_name(run_dir.name + "_figures")

def command(script, *args):
    subprocess.run([sys.executable, str(root / script), *map(str, args)], cwd=root, check=True)


In [ ]:
command("run.py", "bottleneck", "--inflows", "6", "--seeds", "1",
        "--node", node, "--output", run_dir)


In [ ]:
command("evaluate.py", "--experiment", "bottleneck", "--runs", run_dir, "--output", eval_dir)
command("render.py", "--evaluation", eval_dir, "--output", figure_dir)


In [ ]:
import pandas as pd
from IPython.display import Image, display
display(pd.read_csv(eval_dir / "figure_data" / "run_summary.csv"))
for filename in sorted(figure_dir.glob("*.png")):
    display(Image(filename=str(filename)))


## Full stadium experiment

The complete command is below as a template. It uses the included processed data and runs six tunnels, ten seeds and two conditions. Set the worker count for your machine; the launcher also checks memory. For a smaller full-recording example, replace `--full` with `--tunnels E --seeds 1`.

```python
command("run.py", "stadium", "--full", "--workers", "22", "--node", node,
        "--output", root / "outputs" / ("stadium_" + stamp))
```

Use `evaluate.py` and `render.py --animate` afterward as shown in the README. A single seed pair checks execution; ten pairs are needed for the reported study intervals.
